# 03 — Model training

Trains the LightGBM baseline and the PyG GNN side-by-side, compares Pearson r / MAE on the held-out 20% split.

In [ ]:
from models._features import load_cached, build_features
from models.train import _load_inputs

ff = load_cached()
if ff is None:
    labs, events, patients = _load_inputs()
    ff = build_features(labs, events, patients)
print('features:', ff.X.shape, 'targets:', ff.y_hba1c.shape)

In [ ]:
train, test = ff.split(test_frac=0.2, seed=42)
from models.gbm_baseline import train_regressor
gbm_res = train_regressor(train, test, target='hba1c')
gbm_res

In [ ]:
try:
    from models.gnn_hba1c import train_gnn, PYG_AVAILABLE
    if PYG_AVAILABLE:
        gnn_res = train_gnn(train.X, train.y_hba1c, test.X, test.y_hba1c, epochs=15)
        print(gnn_res)
    else:
        print('PyG not installed — skipping GNN cell.')
except ImportError as exc:
    print('PyG import failed:', exc)

In [ ]:
from models.engagement_dropout import train_dropout
drop_res = train_dropout(train, test)
drop_res